<a href="https://colab.research.google.com/github/office268/ipynb/blob/main/ex_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- שלב 0: התקנת ספריות ---
!pip install -q -U langchain langchain-community langchain-google-genai langchain-text-splitters chromadb pypdf

import os
from google.colab import files, userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# הגדרת מפתח ה-API
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# 2. העלאת קובץ מהמכשיר
print("אנא העלה קובץ PDF מהמכשיר שלך:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

In [ ]:
# 3. טעינת הקובץ ופירוק לצ'אנקים של 500
print(f"\nמעבד את הקובץ: {file_name}...")
loader = PyPDFLoader(file_name)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)
texts = text_splitter.split_documents(documents)
print(f"הקובץ פוצל ל-{len(texts)} צ'אנקים.")

In [ ]:
# 4. יצירת ה-Vector Store (בסיס נתונים וקטורי)
print("יוצר Embeddings ושומר בבסיס הנתונים הוקטורי...")
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

In [ ]:
# 5. הגדרת המודל ושרשרת השאלות (LCEL)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template(
    "ענה על השאלה הבאה בהתבסס אך ורק על ההקשר הנתון.\n\n"
    "הקשר:\n{context}\n\nשאלה: {question}"
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 6. הרצת שאילתה לבדיקה
print("\n" + "="*30)
query = input("מה תרצה לשאול על הקובץ שהעלית? ")
if query:
    print("\nחושב על תשובה...")
    response = qa_chain.invoke(query)
    print("\n--- תשובה מה-RAG ---")
    print(response)